# 다양한 sns정보 수집하기


In [ ]:
# 필요한 모듈과 라이브러리 로딩

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import time
import math
import os
import random
import unicodedata
import urllib.request
import urllib
import pandas as pd

In [ ]:
# 사용자에게 필요한 정보 입력받기
print('='*100)
print( ' 인스타그램 해시태그, 이미지 수집 하기 ' )
print('='*100)

v_id = input('1. 인스타그램의 ID를 입력하세요:')
if v_id =='' : 
    v_id = 'jw73721'
v_pw = input('2. 인스타그램의 PW 를 입력하세요:')
if v_pw =='' :
    v_pw = 'rlawjddn11!'

query_txt = input('3. 검색할 해시태그를 입력하세요 ( 예: #강남맛집 ) :')
if query_txt =='':
    query_txt = '#강남맛집'

try: 
    cnt = int(input('4. 수집할 건수는 총 몇건 입니까? (기본값 : 10)'))
except Exception as e:
    cnt = 10
    print('기본값인 10건으로 수집을 진행합니다.')

page_cnt = math.ceil( cnt / 10 )

f_dir = input('5. 파일이 저장될 경로만 쓰세요! (기본경로 : C:\\Users\\ITWILL\\workspace\\seo\\크롤링\\)')
if f_dir == '' :
    f_dir = 'C:\\Users\\ITWILL\\workspace\\seo\\크롤링\\'

In [ ]:
# 결과를 저장할 폴더명과 파일명을 설정하고 폴더 생성하기
n = time.localtime()
s = f'{n.tm_year}-{n.tm_mon}-{n.tm_mday}-{n.tm_hour}-{n.tm_min}-{n.tm_sec}'

sec_name = '인스타그램'
img_dir = f_dir + s + '-' + query_txt + '-' + sec_name + '\\' + 'image'

os.makedirs(img_dir)
os.chdir(img_dir)

fc_name = f_dir + s + '-' + query_txt + '-' + sec_name + '\\' + s + '-' + query_txt + '-' + sec_name + '.csv'
fx_name = f_dir + s + '-' + query_txt + '-' + sec_name + '\\' + s + '-' + query_txt + '-' + sec_name + '.xlsx'

In [ ]:
# 인스타그램 접속 후 자동 로그인

s_time = time.time()
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.instagram.com'
driver.get(url)
time.sleep(random.randrange(2, 5))

driver.maximize_window()

print('\n')
print('요청하신 데이터를 추출 중이오니, 잠시만 기다려주세요')
print('\n')

# ID와 비번 입력 후 로그인 하기
eid = driver.find_element(By.NAME, 'email')
time.sleep(0.5)
for a in v_id : 
    eid.send_keys(a)
    time.sleep(random.random())

epwd = driver.find_element(By.NAME, 'pass')
for b in v_pw:
    epwd.send_keys(b)
    time.sleep(random.random())

epwd.send_keys('\n')

time.sleep(10)


In [ ]:
# 정보 저장 나중에 하기 클릭
driver.find_element(By.CLASS_NAME, 'x1i10hfl ').click()
#driver.find_element(By.XPATH,"//div[text()='나중에 하기']").click()
#/html/body/div[4]/div[1]/div/div[2]/div/div/div/div/div[2]/div/div/div[3]/button[2]

time.sleep(5)

In [ ]:
# 알림정보 나중에 하기 클릭
driver.find_element(By.CLASS_NAME, '_a9-- ').click()
#driver.find_element(By.XPATH,"//button[text()='나중에 하기']").click()

time.sleep(5)

In [ ]:
# 검색할 키워드 입력하기

driver.find_element(By.CSS_SELECTOR, 'svg[aria-label="검색"]').click()

time.sleep(2)

driver.find_element(By.CSS_SELECTOR, 'input[aria-label="입력 검색"]')

time.sleep(2)

element = driver.find_element(By.CSS_SELECTOR,'input[aria-label="입력 검색"]')


for c in query_txt:
    element.send_keys(c)
    time.sleep(0.5)
time.sleep(3)

driver.find_element(By.XPATH, "//a[contains(@href,'/explore/tags/')]").click()

# element.send_keys(Keys.ARROW_DOWN)
# time.sleep(0.2)
# driver.switch_to.active_element.click()

In [ ]:
# 전체 게시물의 원본 url 추출하기

item = []       #인스타그램 URL 주소 저장할 리스트
item2 = []      # 중복값을 제거한 최종 URL 주소를 저장할 리스트

# 자동 스크롤 다운 함수
def scroll_down(driver):
    driver.execute_script("window.scrollTo(0,document.body.scrollHeight);")
    time.sleep(5)

print('요청하신 데이터를 수집중이니 잠시만 기다려주세요!')

a = 1
while ( a <= page_cnt):
    scroll_down(driver)

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    all_a = soup.find('div','x78zum5 xdt5ytf x11lt19s x1n2onr6 xph46j x7x3xai xsybdxg x194l6zq').find_all('a')
    # all_a = soup.find('div','x1n2onr6').find_all('a','x1i10hfl')

    for i in all_a:
        url = i['href']
        item.append(url)
        item2 = pd.Series(item).drop_duplicates()

        if len(item2) >= cnt :
            break
    a += 1
    print('요청하신 데이터를 수집중이니 잠시만 더 기다려 주세요~^^')

In [ ]:
# 추출된 URL 사용하여 전체 URL 완성하기
full_url = []
url_cnt = 1
print(' === 수집될 인스타 그램 주소는 아래와 같습니다. === ')

for x in item2:
    url = 'https://www.instagram.com' + x
    full_url.append(url)
    print(url_cnt, ':', url)
    
    if url_cnt > cnt:
        break
    url_cnt += 1

print('=============================')
print()

In [ ]:
# 각 페이지별로 이미지와 해시태그 수집하기
count = 1
no2 = []
url2 = []
hash2 = []

count = 1

for c in full_url:
    print(c)
    driver.get(c)
    time.sleep(1)

    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')
    tags = soup.find('div','x9f619 x78zum5 xdt5ytf x5yr21d xexx8yu xv54qhq x1l90r2v xf7dkkf x10l6tqk xh8yej3')

    try:
        tags_1 = tags.find_all('a')
    except :
        continue
    else : 
        print(f'{count}번째 게시물의 대표 이미지와 해시태그를 수집합니다.')
        print('게시물 URL:', c)
        no2.append(count)
        url2.append(c)

        #except : 
        img_src = soup.find('div','_aagv').find('img')['src']
        urllib.request.urlretrieve(img_src, str(count) + '.jpg')

        print(img_dir, f'아래에 {count}번째 이미지 저장완료')

        import sys
        bmp_map = dict.fromkeys(range(0x10000, sys.maxunicode + 1), 0xfffd)

        hash_tags=[]
        for d in tags_1 : 
            tags = d.get_text()
            tags_11 = tags.translate(bmp_map)
            tags_2 = unicodedata.normalize('NFC', tags_11)

            if tags_2[0:1] == '#':

                hash_tags.append(tags_2)

        print(hash_tags)
        hash2.append(hash_tags)

    count += 1

In [ ]:
# 수집된 해시태그를 csv , xlsx 형식으로 저장하기
# xls, csv 로 저장하기 위해 데이터 프레임 생성

insta = pd.DataFrame()
insta['번호'] = no2
insta['URL주소'] = url2
insta['해시태그'] = pd.Series(hash2)

# csv 형태로 저장하기
insta.to_csv(fc_name, encoding='utf-8-sig', index=False)

# 엑셀 형태로 저장하기
insta.to_excel(fx_name, index=False)

# 요약정보 출력하기
e_time = time.time( )
t_time = e_time - s_time

print("=" *120)
print("1.총 소요시간: %s 초" %round(t_time,1))
print("2.총 저장 건수: %s 건 " %count)
print("3.csv파일 저장 경로: %s" %fc_name)
print("4.xls파일 저장 경로: %s" %fx_name)
print("5.이미지파일 저장 경로: %s" %img_dir)
print("=" *120)

try:
    driver.quit()
except:
    pass

In [ ]:
# 필요한 모듈 로딩
import excelrd
import pymysql

#파일 불러오기
#xls = excelrd.open_workbook(f'{fx_name}','rb')
xls = excelrd.open_workbook('C:\\Users\\ITWILL\\workspace\\seo\\크롤링\\2026-3-16-16-15-37-#몰디브-인스타그램\\2026-3-16-16-15-37-#몰디브-인스타그램.xlsx','rb')
sheet = xls.sheet_by_name('Sheet1')

In [ ]:
#데이터베이스와의 연결정보 설정
mysql = pymysql.connect(host = '192.168.25.17', port=3306,
                       user = 'my_user',
                       passwd='mysql',
                       db = 'my_testdb',
                       charset = 'utf8')

In [ ]:
#테이블 생성
cursor = mysql.cursor()

query_1 = '''
CREATE TABLE insta(
no TEXT,
url TEXT,
tag TEXT);
'''
cursor.execute(query_1)

In [ ]:
#데이터 입력하기
cursor = mysql.cursor()

query_2 = '''INSERT INTO insta(
no, url, tag) VALUES(%s, %s, %s)'''

for r in range(1, sheet.nrows):
    no = sheet.cell(r, 0).value
    url = sheet.cell(r, 1).value
    tag = sheet.cell(r, 2).value

    values = (no, url, tag)
    cursor.execute(query_2, values)

In [ ]:
#커밋 후 데이터베이스와의 연결종료
cursor.close()
mysql.commit()
mysql.close()

columns = str(sheet.ncols)
rows = str(sheet.nrows)
print(f'총 {rows} 행 {columns} 열의 데이터를 MySQL에 import 했습니다.!')